# PSF-to-AP Flux Ratio as a Zero-Point Proxy per DIA Object

This notebook loads the enriched alert table produced by `03_associateFinkAlerts-RubinVisits.ipynb`
(`src_joined_butler.parquet` preferred, or `src_joined_consdb.parquet` as fallback)
and displays, for the **N top-ranked DIA objects** (sorted by decreasing alert count),
the quantity:

$$\Delta m(t) = 2.5 \log_{10}\!\left(\frac{F_{\rm psf}(t)}{F_{\rm ap}(t)}\right)$$

as a function of relative time $\Delta t$ from the first alert of the object.

### Physical meaning
$\Delta m$ is the magnitude difference between PSF photometry and aperture photometry.
For a perfectly well-calibrated PSF model on a point source, $\Delta m = 0$.
Systematic offsets or time variations reveal PSF-model errors, seeing changes,
or aperture-correction drifts — all of which affect the relative zero-point.

### Error propagation
From the standard propagation of independent Gaussian errors through a logarithm:

$$\sigma_{\Delta m} = \frac{2.5}{\ln 10}\,
\sqrt{\left(\frac{\sigma_{\rm psf}}{F_{\rm psf}}\right)^{\!2}
     + \left(\frac{\sigma_{\rm ap}}{F_{\rm ap}}\right)^{\!2}}$$

### Subplot layout
`u | g | r | i | z | y | all-bands`  (7 panels per DIA object figure).

- x bottom : $\Delta t$ [days] from the first alert
- x top    : calendar date (`YYYY-MM-DD`)
- y        : $\Delta m = 2.5 \log_{10}(F_{\rm psf}/F_{\rm ap})$ [mag]
- Reference line : $\Delta m = 0$

---

- **Author:** Sylvie Dagoret-Campagne  
- **Affiliation:** IJCLab / IN2P3 / CNRS  
- **Last update:** 2026-04-02  
- **Subject:** Fink alert broker — Rubin LSST PSF-vs-AP zero-point proxy

## 1. Imports & configuration

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.time import Time

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
# Enable interactive matplotlib backend (zoom/pan) when ipympl is available
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")
    print("Install with:  pip install ipympl")

In [ ]:
# ── Notebook tag & paths ──────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_04e"
DIR_DATA_IN = "data_FINK_BLOCK_LC_03"  # input data written by notebook 03
DIR_FIGS = f"figs_{NB_TAG}"  # output figures for this notebook
os.makedirs(DIR_FIGS, exist_ok=True)

# ── Source files (butler preferred; fallback to consdb) ───────────────────────
FILE_BUTLER = os.path.join(DIR_DATA_IN, "src_joined_butler.parquet")
FILE_CONSDB = os.path.join(DIR_DATA_IN, "src_joined_consdb.parquet")

# ── Number of top-ranked DIA objects to plot ──────────────────────────────────
N_TOP = 50  # <── change here

# ── Band definitions ──────────────────────────────────────────────────────────
BANDS = list("ugrizy")
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}

# ── Conversion factor: 2.5 / ln(10) used in error propagation ────────────────
MAGFAC = 2.5 / np.log(10.0)  # ≈ 1.0857

# ── Matplotlib global settings ────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)

# ── Column names expected in the parquet file ─────────────────────────────────
COL_OBJ = "r:diaObjectId"
COL_MJD = "r:midpointMjdTai"
COL_BAND = "r:band"
COL_PSF = "r:psfFlux"
COL_PSFERR = "r:psfFluxErr"
COL_AP = "r:apFlux"
COL_APERR = "r:apFluxErr"


def savefig(name):
    """Save current figure as both PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  → saved {name}.{{pdf,png}}")


print(f"Input directory  : {os.path.abspath(DIR_DATA_IN)}")
print(f"Figure directory : {os.path.abspath(DIR_FIGS)}")
print(f"N_TOP            : {N_TOP}")
print(f"MAGFAC (2.5/ln10): {MAGFAC:.6f}")

## 2. Load data

In [ ]:
# Butler preferred; fallback to consDb
if os.path.exists(FILE_BUTLER):
    df = pd.read_parquet(FILE_BUTLER)
    src_label = "butler"
    print(f"Loaded butler file : {FILE_BUTLER}")
elif os.path.exists(FILE_CONSDB):
    df = pd.read_parquet(FILE_CONSDB)
    src_label = "consdb"
    print(f"Butler file not found. Loaded consDb file : {FILE_CONSDB}")
else:
    raise FileNotFoundError(
        f"Neither {FILE_BUTLER} nor {FILE_CONSDB} found. "
        "Run notebook 03_associateFinkAlerts-RubinVisits.ipynb first."
    )

print(f"Shape : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Label : {src_label}")

## 3. Schema inspection & column validation

In [ ]:
print("Columns in loaded table:")
print(df.dtypes.to_string())

In [ ]:
# Validate that all required columns are present
required_cols = [COL_OBJ, COL_MJD, COL_BAND, COL_PSF, COL_PSFERR, COL_AP, COL_APERR]
missing_cols = [c for c in required_cols if c not in df.columns]

if missing_cols:
    raise KeyError(
        f"Missing required columns: {missing_cols}\n"
        "Both psfFlux and apFlux (with their errors) are mandatory for this notebook."
    )

print("All required columns present (psfFlux + apFlux + errors).")

## 4. Rank DIA objects by number of alerts (decreasing)

In [ ]:
# Count alerts per diaObjectId and sort descending
alert_counts = df.groupby(COL_OBJ).size().rename("n_alerts").sort_values(ascending=False)

print(f"Total unique DIA objects : {len(alert_counts):,}")
print(f"Top {N_TOP} by alert count:")
print(alert_counts.head(N_TOP).to_string())

In [ ]:
# Select top-N objects and ensure MJD is float
top_objects = alert_counts.head(N_TOP).index.tolist()
df_top = df[df[COL_OBJ].isin(top_objects)].copy()
df_top[COL_MJD] = df_top[COL_MJD].astype(float)

print(f"Rows kept for top-{N_TOP} objects : {len(df_top):,}")

## 5. Utility functions

### Zero-point proxy definition

$$\Delta m = 2.5 \log_{10}\!\left(\frac{F_{\rm psf}}{F_{\rm ap}}\right)$$

### Error propagation

Differentiating $\Delta m$ with respect to the two independent flux measurements:

$$\frac{\partial\,\Delta m}{\partial F_{\rm psf}} = \frac{2.5}{\ln 10}\,\frac{1}{F_{\rm psf}}, 
\qquad
\frac{\partial\,\Delta m}{\partial F_{\rm ap}}  = -\frac{2.5}{\ln 10}\,\frac{1}{F_{\rm ap}}$$

$$\Rightarrow \quad
\sigma_{\Delta m} = \frac{2.5}{\ln 10}\,
\sqrt{\left(\frac{\sigma_{\rm psf}}{F_{\rm psf}}\right)^{\!2}
     +\left(\frac{\sigma_{\rm ap}}{F_{\rm ap}}\right)^{\!2}}$$

Points where either flux is non-positive (non-detection or negative background residual)
are masked to NaN and excluded from statistics.

In [ ]:
def mjd_to_datestr(mjd_array):
    """
    Convert an array of MJD (float) values to ISO date strings 'YYYY-MM-DD'.
    Uses astropy.time.Time for accuracy.
    """
    t = Time(np.asarray(mjd_array, dtype=float), format="mjd", scale="tai")
    return [tt.strftime("%Y-%m-%d") for tt in t]


def compute_zp_proxy(psf_flux, psf_err, ap_flux, ap_err):
    """
    Compute the PSF-to-AP magnitude difference as a zero-point proxy.

    Definition
    ----------
    dm = 2.5 * log10(psfFlux / apFlux)

    Error propagation (independent Gaussian errors)
    ------------------------------------------------
    sigma_dm = (2.5 / ln10) * sqrt( (sigma_psf / psfFlux)^2
                                   + (sigma_ap  / apFlux )^2 )

    Points where psfFlux <= 0 or apFlux <= 0 are set to NaN.

    Parameters
    ----------
    psf_flux : array-like  — psfFlux values (nJy)
    psf_err  : array-like  — psfFluxErr values (nJy)
    ap_flux  : array-like  — apFlux values (nJy)
    ap_err   : array-like  — apFluxErr values (nJy)

    Returns
    -------
    dm       : ndarray  — 2.5 * log10(psfFlux / apFlux)  [mag]
    dm_err   : ndarray  — 1-sigma uncertainty on dm  [mag]
    valid    : ndarray  — boolean mask of usable (finite, positive) points
    dm_mean  : float    — mean of dm over valid points
    dm_std   : float    — RMS scatter of dm over valid points
    """
    f_psf = np.asarray(psf_flux, dtype=float)
    fe_psf = np.asarray(psf_err, dtype=float)
    f_ap = np.asarray(ap_flux, dtype=float)
    fe_ap = np.asarray(ap_err, dtype=float)

    # Valid mask: both fluxes must be finite and strictly positive
    valid = (
        np.isfinite(f_psf)
        & (f_psf > 0)
        & np.isfinite(f_ap)
        & (f_ap > 0)
        & np.isfinite(fe_psf)
        & np.isfinite(fe_ap)
    )

    dm = np.full(len(f_psf), np.nan)
    dm_err = np.full(len(f_psf), np.nan)

    if valid.sum() == 0:
        return dm, dm_err, valid, np.nan, np.nan

    # Zero-point proxy: dm = 2.5 * log10(psf / ap)
    dm[valid] = 2.5 * np.log10(f_psf[valid] / f_ap[valid])

    # Error propagation: sigma_dm = MAGFAC * sqrt( (sig_psf/f_psf)^2 + (sig_ap/f_ap)^2 )
    dm_err[valid] = MAGFAC * np.sqrt((fe_psf[valid] / f_psf[valid]) ** 2 + (fe_ap[valid] / f_ap[valid]) ** 2)

    dm_mean = float(np.nanmean(dm[valid]))
    dm_std = float(np.nanstd(dm[valid]))

    return dm, dm_err, valid, dm_mean, dm_std


def _add_date_axis(ax, dt_array, t0_mjd):
    """
    Add a secondary x-axis on top of *ax* showing calendar dates.

    Strategy: pick at most 5 evenly spaced tick positions from the data range,
    convert their MJD to ISO date strings, and attach them as a twin top axis.

    Parameters
    ----------
    ax       : matplotlib Axes
    dt_array : 1-D array of Δt [days] already plotted on ax
    t0_mjd   : float — reference MJD (first alert of the object)
    """
    dt_finite = dt_array[np.isfinite(dt_array)]
    if len(dt_finite) == 0:
        return

    dt_min, dt_max = float(dt_finite.min()), float(dt_finite.max())
    if dt_max <= dt_min:
        tick_dt = np.array([dt_min])
    else:
        n_ticks = min(5, max(2, int((dt_max - dt_min) / 10) + 2))
        tick_dt = np.linspace(dt_min, dt_max, n_ticks)

    tick_dates = mjd_to_datestr(t0_mjd + tick_dt)

    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim())
    ax2.set_xticks(tick_dt)
    ax2.set_xticklabels(tick_dates, rotation=35, ha="left", fontsize=7)
    ax2.tick_params(axis="x", length=3)


print("Utility functions defined (compute_zp_proxy, _add_date_axis).")

## 6. Plotting function: ZP-proxy light curve per DIA object

For each DIA object, produce 7 subplots:
- **Subplots 1–6** : one per band (`u g r i z y`)
- **Subplot 7** : all bands superimposed

Each panel shows $\Delta m(t) = 2.5\log_{10}(F_{\rm psf}/F_{\rm ap})$ with error bars
and a dashed reference line at $\Delta m = 0$.
The title reports mean and RMS of $\Delta m$ for that band.

In [ ]:
def plot_zp_proxy_object(obj_id, df_obj, axes):
    """
    Plot the PSF-vs-AP magnitude difference (ZP proxy) for one DIA object.

    Parameters
    ----------
    obj_id : int / str     — diaObjectId
    df_obj : pd.DataFrame  — rows for this object, all bands
    axes   : list of 7 matplotlib Axes
                            [ax_u, ax_g, ax_r, ax_i, ax_z, ax_y, ax_all]

    Returns
    -------
    t0_date : str  — ISO date string of the first alert (used in the figure title)
    """
    # Reference time: MJD of the very first alert across all bands
    t0_mjd = df_obj[COL_MJD].min()
    t0_date = mjd_to_datestr([t0_mjd])[0]

    # ── Per-band subplots (indices 0..5) ──────────────────────────────────────
    for idx, band in enumerate(BANDS):
        ax = axes[idx]
        mask = df_obj[COL_BAND] == band
        df_b = df_obj[mask].sort_values(COL_MJD)

        if len(df_b) == 0:
            # No data in this band for this object
            ax.text(
                0.5,
                0.5,
                "no data",
                ha="center",
                va="center",
                transform=ax.transAxes,
                color="gray",
                fontsize=8,
            )
            ax.set_title(f"band {band}", fontsize=9)
            continue

        dt = df_b[COL_MJD].values - t0_mjd  # relative time [days]

        dm, dm_err, valid, dm_mean, dm_std = compute_zp_proxy(
            df_b[COL_PSF].values,
            df_b[COL_PSFERR].values,
            df_b[COL_AP].values,
            df_b[COL_APERR].values,
        )

        n_valid = int(valid.sum())
        n_invalid = len(df_b) - n_valid
        color = BAND_COLORS.get(band, "k")

        # Plot valid points with error bars
        ax.errorbar(
            dt[valid],
            dm[valid],
            yerr=dm_err[valid],
            fmt="o",
            ms=4,
            color=color,
            ecolor=color,
            elinewidth=0.8,
            capsize=2,
            alpha=0.85,
            label=f"{band}  <Δm>={dm_mean:.3f}  σ={dm_std:.3f}",
        )

        # Mark non-positive-flux points that were masked
        if n_invalid > 0:
            ax.scatter(
                dt[~valid],
                np.zeros(n_invalid),  # place on reference line
                marker="x",
                s=20,
                color=color,
                alpha=0.5,
                label=f"{n_invalid} masked (F≤0)",
                zorder=2,
            )

        # Reference line: Δm = 0 means psf == ap
        ax.axhline(0.0, color="k", lw=0.7, ls="--", alpha=0.5)

        # Title: band label + statistics
        title_str = f"band {band}  |  <Δm>={dm_mean:.3f}  σ={dm_std:.3f}"
        if not np.isnan(dm_mean):
            title_str += f"  (n={n_valid})"
        ax.set_title(title_str, fontsize=9)
        ax.set_ylabel(r"$\Delta m = 2.5\,\log_{10}(F_{\rm psf}/F_{\rm ap})$ [mag]", fontsize=8)
        ax.legend(fontsize=7, loc="best")

        # Top x-axis: calendar dates derived from Δt
        _add_date_axis(ax, dt, t0_mjd)

    # ── Combined panel (subplot 7, index 6) ───────────────────────────────────
    ax_all = axes[-1]

    for band in BANDS:
        mask = df_obj[COL_BAND] == band
        df_b = df_obj[mask].sort_values(COL_MJD)
        if len(df_b) == 0:
            continue

        dt = df_b[COL_MJD].values - t0_mjd
        dm, dm_err, valid, dm_mean, dm_std = compute_zp_proxy(
            df_b[COL_PSF].values,
            df_b[COL_PSFERR].values,
            df_b[COL_AP].values,
            df_b[COL_APERR].values,
        )
        color = BAND_COLORS.get(band, "k")

        if valid.sum() == 0:
            continue

        ax_all.errorbar(
            dt[valid],
            dm[valid],
            yerr=dm_err[valid],
            fmt="o",
            ms=3,
            color=color,
            ecolor=color,
            elinewidth=0.7,
            capsize=2,
            alpha=0.70,
            label=f"{band} <Δm>={dm_mean:.3f} σ={dm_std:.3f}",
        )

    ax_all.axhline(0.0, color="k", lw=0.7, ls="--", alpha=0.5)
    ax_all.set_title("all bands", fontsize=9)
    ax_all.set_ylabel(r"$\Delta m$ [mag]", fontsize=8)
    ax_all.legend(fontsize=7, ncol=2, loc="best")
    _add_date_axis(ax_all, df_obj[COL_MJD].values - t0_mjd, t0_mjd)

    return t0_date


print("plot_zp_proxy_object() defined.")

## 7. Main loop: ZP-proxy light curves for top-N DIA objects

In [ ]:
n_subplots = len(BANDS) + 1  # 6 individual bands + 1 combined panel

for rank, obj_id in enumerate(top_objects):
    df_obj = df_top[df_top[COL_OBJ] == obj_id].sort_values(COL_MJD)
    n_alerts = len(df_obj)

    # Retrieve the DDF field name (take first non-null unique value)
    field_vals = df_obj["field"].dropna().unique() if "field" in df_obj.columns else []
    field_str = str(field_vals[0]) if len(field_vals) > 0 else "?"

    fig, axes = plt.subplots(
        1,
        n_subplots,
        figsize=(3.2 * n_subplots, 4.5),
        sharey=False,
        constrained_layout=True,
    )

    t0_date = plot_zp_proxy_object(obj_id, df_obj, axes)

    # Common bottom x-label on all subplots
    for ax in axes:
        ax.set_xlabel("Δt [days] from first alert", fontsize=8)

    fig.suptitle(
        f"rank #{rank + 1}  |  diaObjectId={obj_id}  |  field={field_str}  "
        f"|  N={n_alerts} alerts  |  t₀={t0_date}\n"
        r"$\Delta m = 2.5\,\log_{10}(F_{\rm psf}/F_{\rm ap})$  "
        "— reference line: $\Delta m = 0$",
        fontsize=10,
        y=1.05,
    )

    savefig(f"zp_proxy_{src_label}_rank{rank + 1:02d}_obj{obj_id}")
    plt.show()
    # plt.close(fig)

## 8. Summary table: per-object, per-band ZP-proxy statistics

For each top-ranked object and each band we report:
- `dm_mean_{band}` : mean of $\Delta m$ [mag]
- `dm_std_{band}`  : RMS scatter of $\Delta m$ [mag]
- `n_valid_{band}` : number of epochs with both fluxes positive

A non-zero `dm_mean` signals a systematic PSF-vs-AP offset in that band;
a large `dm_std` suggests temporal variability in the PSF model or aperture correction.

In [ ]:
records = []

for rank, obj_id in enumerate(top_objects):
    df_obj = df_top[df_top[COL_OBJ] == obj_id]
    n_total = len(df_obj)
    t0_mjd = df_obj[COL_MJD].min()
    t0_date = mjd_to_datestr([t0_mjd])[0]

    field_vals = df_obj["field"].dropna().unique() if "field" in df_obj.columns else []
    field_str = str(field_vals[0]) if len(field_vals) > 0 else "?"

    row = {
        "rank": rank + 1,
        "diaObjectId": obj_id,
        "field": field_str,
        "n_alerts": n_total,
        "t0_date": t0_date,
    }

    for band in BANDS:
        df_b = df_obj[df_obj[COL_BAND] == band]
        if len(df_b) == 0:
            row[f"dm_mean_{band}"] = np.nan
            row[f"dm_std_{band}"] = np.nan
            row[f"n_valid_{band}"] = 0
            continue

        _, _, valid, dm_mean, dm_std = compute_zp_proxy(
            df_b[COL_PSF].values,
            df_b[COL_PSFERR].values,
            df_b[COL_AP].values,
            df_b[COL_APERR].values,
        )
        row[f"dm_mean_{band}"] = round(dm_mean, 4) if not np.isnan(dm_mean) else np.nan
        row[f"dm_std_{band}"] = round(dm_std, 4) if not np.isnan(dm_std) else np.nan
        row[f"n_valid_{band}"] = int(valid.sum())

    records.append(row)

df_summary = pd.DataFrame(records)
print("Summary table of ZP-proxy statistics per object & band:")
display(df_summary)

In [ ]:
# Save summary table to CSV
summary_path = os.path.join(DIR_FIGS, f"zp_proxy_summary_{src_label}.csv")
df_summary.to_csv(summary_path, index=False)
print(f"Summary saved to {summary_path}")

## 9. Overview plot: ZP-proxy mean per band across all top-N objects

A quick heatmap-style visualisation showing `dm_mean` per (object, band),
useful for spotting systematic band-dependent offsets at a glance.

In [ ]:
# Build a matrix: rows = objects (ranked), columns = bands
mean_cols = [f"dm_mean_{b}" for b in BANDS]
std_cols = [f"dm_std_{b}" for b in BANDS]

mat_mean = df_summary[mean_cols].values.astype(float)  # shape (N_objects, 6)
mat_std = df_summary[std_cols].values.astype(float)

obj_labels = [f"#{r}  {oid}" for r, oid in zip(df_summary["rank"], df_summary["diaObjectId"])]

fig, axes_ov = plt.subplots(1, 2, figsize=(14, max(4, 0.35 * len(top_objects) + 1)), constrained_layout=True)

for ax_ov, mat, title in zip(
    axes_ov,
    [mat_mean, mat_std],
    [r"$\langle\Delta m\rangle$ per (object, band) [mag]", r"$\sigma(\Delta m)$ per (object, band) [mag]"],
):
    vmax = np.nanpercentile(np.abs(mat), 95) if not np.all(np.isnan(mat)) else 0.1
    vmax = max(vmax, 1e-4)  # avoid zero-range colour scale

    im = ax_ov.imshow(
        mat,
        aspect="auto",
        cmap="RdBu_r" if "mean" in title else "viridis",
        vmin=-vmax if "mean" in title else 0,
        vmax=vmax,
        interpolation="nearest",
    )
    plt.colorbar(im, ax=ax_ov, label="mag", fraction=0.03, pad=0.02)

    ax_ov.set_xticks(range(len(BANDS)))
    ax_ov.set_xticklabels(BANDS, fontsize=9)
    ax_ov.set_yticks(range(len(obj_labels)))
    ax_ov.set_yticklabels(obj_labels, fontsize=7)
    ax_ov.set_xlabel("Band", fontsize=9)
    ax_ov.set_title(title, fontsize=9)

    # Annotate cells with numeric value (skip NaN)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            if not np.isnan(mat[i, j]):
                ax_ov.text(
                    j,
                    i,
                    f"{mat[i, j]:.3f}",
                    ha="center",
                    va="center",
                    fontsize=6,
                    color="k",
                )

fig.suptitle(
    f"ZP-proxy overview — top {len(top_objects)} DIA objects — {src_label}",
    fontsize=11,
)
savefig(f"zp_proxy_overview_{src_label}")
plt.show()